:meth:`SeqOpt.run` scores a variant as a *genome*, a sparse ``{1-based position: target amino acid}`` mapping over one wild-type, and feeds each feasibility callable into the penalty term of its fitness function. :meth:`DesignConstraints.as_predicate` adapts the shared limits to exactly that ``genome -> bool`` shape, so the same constraint set drives the optimizer and :meth:`DesignConstraints.check`.

In [1]:
import pandas as pd
import aaanalysis as aa
aa.options["verbose"] = False

df_seq = aa.load_dataset(name="DOM_GSEC", n=5)
seq = df_seq["sequence"].iloc[0]
tmd_start, tmd_stop = int(df_seq["tmd_start"].iloc[0]), int(df_seq["tmd_stop"].iloc[0])
pos_a, pos_b, pos_c = tmd_start + 3, tmd_start + 6, tmd_start + 9

# parent is not stored on the object, so it is passed to as_predicate()
dc = aa.DesignConstraints(immutable_positions=[tmd_start, tmd_stop],
                          permitted_substitutions=["A", "L", "V", "I"],
                          n_mut_max=2)
is_feasible = dc.as_predicate(parent=seq)

genomes = {"one_sub": {pos_a: "A"},
           "two_subs": {pos_a: "A", pos_b: "L"},
           "three_subs": {pos_a: "A", pos_b: "L", pos_c: "V"},
           "anchor_hit": {tmd_start: "A"},
           "banned_aa": {pos_a: "W"}}
df_genome = pd.DataFrame([dict(genome=name, n_subs=len(g), is_feasible=is_feasible(g))
                          for name, g in genomes.items()])
aa.display_df(df_genome, n_rows=10, show_shape=True)

DataFrame shape: (5, 3)


,genome,n_subs,is_feasible
1,one_sub,1,True
2,two_subs,2,True
3,three_subs,3,False
4,anchor_hit,1,False
5,banned_aa,1,False


`parent` is optional: with a parent stored on the object, `as_predicate()` needs no argument, and passing one per call overrides it, so a single constraint set can be adapted to a second protein. The returned callable is what `SeqOpt.run(constraints=[is_feasible])` consumes; `SeqOpt.run` also accepts the `DesignConstraints` object itself as `constraints`, which builds this predicate internally.

In [2]:
# Same limits, parent stored on the object
dc = aa.DesignConstraints(immutable_positions=[tmd_start, tmd_stop],
                          permitted_substitutions=["A", "L", "V", "I"],
                          n_mut_max=2,
                          parent=seq)
is_feasible_stored = dc.as_predicate()

# A per-call parent overrides the stored one (here: the second protein)
seq_other = df_seq["sequence"].iloc[1]
is_feasible_other = dc.as_predicate(parent=seq_other)

df_genome = pd.DataFrame([dict(genome=name,
                               stored_parent=is_feasible_stored(g),
                               other_parent=is_feasible_other(g),
                               matches_check=is_feasible_stored(g) == is_feasible(g))
                          for name, g in genomes.items()])
aa.display_df(df_genome, n_rows=10, show_shape=True)

DataFrame shape: (5, 4)


,genome,stored_parent,other_parent,matches_check
1,one_sub,True,True,True
2,two_subs,True,True,True
3,three_subs,False,False,True
4,anchor_hit,False,False,True
5,banned_aa,False,False,True
